In [1]:
import logging
import platform
import random
import subprocess
import time
from os import getenv

import fs_structs
from dotenv import load_dotenv

from distributed_processing.async_result import gather
from distributed_processing.utils import fsclient

In [2]:
load_dotenv()
NS_PATH = getenv("NS_PATH")
MASTER_QUEUE = getenv("MASTER_QUEUE")
LAUNCH_SERVER = False

MASTER_QUEUE

'node_1'

In [3]:
# logging.getLogger("distributed_processing").setLevel(logging.DEBUG)
# logging.getLogger("distributed_processing.filesystem_connector").setLevel(logging.DEBUG)

In [4]:
if LAUNCH_SERVER:
    PYTHON_INTERPRETER = getenv("PYTHON_INTERPRETER")
    MULTI_WORKER_SCRIPT = getenv("MULTI_WORKER_SCRIPT")
    server_process = subprocess.Popen(
        [PYTHON_INTERPRETER, MULTI_WORKER_SCRIPT],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        shell=False,
    )
    time.sleep(10)

In [5]:
# NBVAL_CHECK_OUTPUT
"""
fs_connector = FileSystemConnector(NS_PATH)
fs_connector.with_watchdog=True
fs_connector.pop_watchdog_timeout = 10

client = Client(DummySerializer(), fs_connector, check_registry="cache")
"""

client = fsclient(NS_PATH)
client.timeout = 70 # el watchdog por defecto es 60s

Client with id: fs_client_1
Results queue: fs_client_1_responses


In [6]:
#MASTER_QUEUE = client.to_requests_queue(MASTER_QUEUE)

In [7]:
client.update_registry_cache()

In [8]:
ns = fs_structs.structs.FSNamespace(NS_PATH)

In [9]:
# NBVAL_CHECK_OUTPUT
ns.registry.keys()

['method_queues_add',
 'method_queues_cleanup',
 'method_queues_create_worker',
 'method_queues_dic',
 'method_queues_div',
 'method_queues_eval_py_function',
 'method_queues_hola',
 'method_queues_info',
 'method_queues_kill_all_processes',
 'method_queues_kill_process',
 'method_queues_kill_processes',
 'method_queues_list_processes',
 'method_queues_lista',
 'method_queues_mul',
 'method_queues_sleep',
 'method_queues_tupla',
 'nclients',
 'nservers',
 'workers_queue_requests_cola_0',
 'workers_queue_requests_cola_1',
 'workers_queue_requests_cola_2',
 'workers_queue_requests_node_1',
 'workers_queue_requests_py_eval']

In [10]:
client._registry

{'methods': {'add': ['requests_cola_1'],
  'cleanup': ['requests_node_1'],
  'create_worker': ['requests_node_1'],
  'dic': ['requests_cola_1'],
  'div': ['requests_cola_1'],
  'eval_py_function': ['requests_py_eval'],
  'hola': ['requests_cola_2'],
  'info': ['requests_cola_0'],
  'kill_all_processes': ['requests_node_1'],
  'kill_process': ['requests_node_1'],
  'kill_processes': ['requests_node_1'],
  'list_processes': ['requests_node_1'],
  'lista': ['requests_cola_1'],
  'mul': ['requests_cola_1'],
  'sleep': ['requests_cola_1'],
  'tupla': ['requests_cola_1']},
 'workers': {'requests_cola_0': ['fs_server_3', 'fs_server_1', 'fs_server_2'],
  'requests_cola_1': ['fs_server_3', 'fs_server_1', 'fs_server_2'],
  'requests_cola_2': ['fs_server_3', 'fs_server_1', 'fs_server_2'],
  'requests_node_1': ['node_1'],
  'requests_py_eval': ['fs_server_3', 'fs_server_1', 'fs_server_2']}}

In [11]:
client.registry()

{'methods': {'add': ['cola_1'],
  'cleanup': ['node_1'],
  'create_worker': ['node_1'],
  'dic': ['cola_1'],
  'div': ['cola_1'],
  'eval_py_function': ['py_eval'],
  'hola': ['cola_2'],
  'info': ['cola_0'],
  'kill_all_processes': ['node_1'],
  'kill_process': ['node_1'],
  'kill_processes': ['node_1'],
  'list_processes': ['node_1'],
  'lista': ['cola_1'],
  'mul': ['cola_1'],
  'sleep': ['cola_1'],
  'tupla': ['cola_1']},
 'workers': {'cola_0': ['fs_server_3', 'fs_server_1', 'fs_server_2'],
  'cola_1': ['fs_server_3', 'fs_server_1', 'fs_server_2'],
  'cola_2': ['fs_server_3', 'fs_server_1', 'fs_server_2'],
  'node_1': ['node_1'],
  'py_eval': ['fs_server_3', 'fs_server_1', 'fs_server_2']}}

In [12]:
client.all_queues_for_method("add")

['cola_1']

In [13]:
# create workers with default pop_watchdog_timeout = 60 

z1 = client.rpc_sync("create_worker", ["worker1"], queue=MASTER_QUEUE)
z2 = client.rpc_sync("create_worker", ["worker1"], queue=MASTER_QUEUE)
z3 = client.rpc_sync("create_worker", ["worker1"], queue=MASTER_QUEUE)

(z1, z2, z3)

((41904, 'worker1', 'fs_server_4'),
 (15892, 'worker1', 'fs_server_5'),
 (20756, 'worker1', 'fs_server_6'))

In [14]:
ps = client.rpc_sync("list_processes", [], queue=MASTER_QUEUE)
ps

[(37780, 'worker1', 'fs_server_1'),
 (12728, 'worker1', 'fs_server_2'),
 (22440, 'worker1', 'fs_server_3'),
 (41904, 'worker1', 'fs_server_4'),
 (15892, 'worker1', 'fs_server_5'),
 (20756, 'worker1', 'fs_server_6')]

In [15]:
# NBVAL_CHECK_OUTPUT

len(ps)

6

In [16]:
# NBVAL_CHECK_OUTPUT

[(w, q) for _, w, q in ps]

[('worker1', 'fs_server_1'),
 ('worker1', 'fs_server_2'),
 ('worker1', 'fs_server_3'),
 ('worker1', 'fs_server_4'),
 ('worker1', 'fs_server_5'),
 ('worker1', 'fs_server_6')]

In [17]:
# NBVAL_CHECK_OUTPUT

client.rpc_sync("kill_process", [z2], queue=MASTER_QUEUE)

False

In [18]:
client.rpc_sync("list_processes", [], queue=MASTER_QUEUE)

[(37780, 'worker1', 'fs_server_1'),
 (12728, 'worker1', 'fs_server_2'),
 (22440, 'worker1', 'fs_server_3'),
 (41904, 'worker1', 'fs_server_4'),
 (15892, 'worker1', 'fs_server_5'),
 (20756, 'worker1', 'fs_server_6')]

In [19]:
# NBVAL_CHECK_OUTPUT

wps = [pid for pid, w, q in client.rpc_sync("list_processes", [], queue=MASTER_QUEUE)]
len(wps)

6

In [20]:
client.rpc_sync("kill_all_processes", [], queue=MASTER_QUEUE)

[(True, False),
 (True, False),
 (True, False),
 (True, False),
 (True, False),
 (True, False)]

In [21]:
# NBVAL_CHECK_OUTPUT

wps = [pid for pid, w, q in client.rpc_sync("list_processes", [], queue=MASTER_QUEUE)]
len(wps)

0

In [ ]:
# create workers with pop_watchdog_timeout = 20 
# first argument, worker_id, if None, sets the worker_id automatically

z1 = client.rpc_sync("create_worker", ["worker1", [None, 20]], queue=MASTER_QUEUE)
z2 = client.rpc_sync("create_worker", ["worker1", [], {"watchdog_timeout":20}], queue=MASTER_QUEUE)

# TypeError, multiple values for worker_id. Returns None, None, None. Process is killed. 
z3 = client.rpc_sync("create_worker", ["worker1", [None], {"worker_id":None, "watchout_timeout":20}], queue=MASTER_QUEUE)

z3

(None, None, None)

In [23]:
client.rpc_sync("list_processes", [], queue=MASTER_QUEUE)

[(37596, 'worker1', 'fs_server_7'), (23956, 'worker1', 'fs_server_8')]

In [24]:
# NBVAL_CHECK_OUTPUT

wps = [pid for pid, w, q in client.rpc_sync("list_processes", [], queue=MASTER_QUEUE)]
len(wps)

2

In [25]:
client.update_registry_cache()

In [26]:
r = client.registry(True)
r

{'methods': {'add': ['cola_1'],
  'cleanup': ['node_1'],
  'create_worker': ['node_1'],
  'dic': ['cola_1'],
  'div': ['cola_1'],
  'eval_py_function': ['py_eval'],
  'hola': ['cola_2'],
  'info': ['cola_0'],
  'kill_all_processes': ['node_1'],
  'kill_process': ['node_1'],
  'kill_processes': ['node_1'],
  'list_processes': ['node_1'],
  'lista': ['cola_1'],
  'mul': ['cola_1'],
  'sleep': ['cola_1'],
  'tupla': ['cola_1']},
 'workers': {'cola_0': ['fs_server_8', 'fs_server_7'],
  'cola_1': ['fs_server_8', 'fs_server_7'],
  'cola_2': ['fs_server_8', 'fs_server_7'],
  'node_1': ['node_1'],
  'py_eval': ['fs_server_8', 'fs_server_7']}}

In [27]:
#time.sleep(5)

In [28]:
client.rpc_sync("add", [1, 2])

3

In [29]:
# NBVAL_CHECK_OUTPUT

def f():
    return globals()

a = client.rpc_sync_fn(f, [])

In [30]:
n = 20
ws = []
for worker in client.all_workers_for_method("add"):
    w = client.rpc_async("add", [1, n], queue=worker, ack=True)
    rq = client.to_requests_queue_ref(w.queue)
    print(ns.variable(rq)[:])

    ws.append(w)
    n+=3

client.pending

[{'method': 'add', 'is_notification': False, 'id': 'fs_client_1:17', 'ack': True, 'args': [1, 20], 'timing': {'request_sent': 1774951053.3042057}}]
[{'method': 'add', 'is_notification': False, 'id': 'fs_client_1:18', 'ack': True, 'args': [1, 23], 'timing': {'request_sent': 1774951053.330677}}]


{'fs_client_1:17': 1774951053.3244796, 'fs_client_1:18': 1774951053.3500605}

In [31]:
for w in ws:
    try:
        w.get(2)
    except:
        rq = client.to_requests_queue_ref(w.queue)
        print(w.queue)
        print(ns.variable(rq)[:])



fs_server_7
[{'method': 'add', 'is_notification': False, 'id': 'fs_client_1:17', 'ack': True, 'args': [1, 20], 'timing': {'request_sent': 1774951053.3042057}}]


In [32]:
client.pending

{'fs_client_1:17': 1774951053.3244796}

In [33]:
# tengo que mirarlo. casi seguro es un problema con el watchdog del worker. alguno de ellos se queda 
# pillado. 

[x.get() for x in ws]

[21, 24]

In [34]:
client.pending

{}

In [35]:
# NBVAL_CHECK_OUTPUT

client.rpc_sync("add", [1, 20], queue=r["methods"]["add"][0])

21

In [36]:
# NBVAL_CHECK_OUTPUT

n = 20
w = []
for worker in client.all_workers_for_method("add"):
    w.append(client.rpc_sync("add", [1, n], queue=worker))
    n+=3

w

[21, 24]

In [37]:
client.all_workers_for_method("sleep")

['fs_server_7', 'fs_server_8']

In [38]:
n = 20
w = []
t = 5
start = time.time()
for worker in client.all_workers_for_method("sleep"):
    w.append(client.rpc_async("sleep", [t], queue=worker))

[x.get() for x in w]

stop = time.time()

print((stop-start)-t)

0.27778124809265137


In [39]:
# NBVAL_CHECK_OUTPUT

y = client.rpc_async("add", [1, 20], queue=client.all_workers_for_method("add")[0])

In [40]:
y.queue

'fs_server_7'

In [41]:
y = client.rpc_async("add", [1, 0])

In [42]:
# NBVAL_CHECK_OUTPUT

y.get()

1

In [43]:
# NBVAL_CHECK_OUTPUT

client.check_registry = "always"

try:
    y = client.rpc_async("fake", [1, 0])
except Exception as e:
    print(e)

'method_queues_fake'


In [44]:
# NBVAL_CHECK_OUTPUT

client.check_registry = "never"  # use queue client.default_queue, by default "default"
client.set_default_queue("cola_1")

y = client.rpc_async("fake", [1, 0])

try:
    y.get()
except Exception as e:
    print(e)


Error -32601 : The method does not exist/is not available.

 NoneType: None



In [45]:
# NBVAL_CHECK_OUTPUT

client.check_registry = "never"
client.set_default_queue("cola_1")

y = client.rpc_async("fake", [1, 0])

y.safe_get(default="Esto es un error")

'Esto es un error'

In [46]:
client.check_registry = "cache"

In [47]:
x = client.rpc_async("div", [1, 0])

In [48]:
try:
    x.get()
except Exception as e:
    print(e)

Error -3260 : Undefined error

 Traceback (most recent call last):
  File "\\ntcimdavinfo2\fondos\arquitectura_gestora\desarrollo\py_distributed_processing\distributed_processing\worker.py", line 364, in process_single_request
    result = self._exec_method_in_queue(dispatched_to, request["method"], args, kwargs)
  File "\\ntcimdavinfo2\fondos\arquitectura_gestora\desarrollo\py_distributed_processing\distributed_processing\worker.py", line 306, in _exec_method_in_queue
    return funcs[method](*args, **kwargs)
           ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "G:\arquitectura_gestora\desarrollo\py_distributed_processing\tests\fs_worker_multi.py", line 32, in div
    return x / y
           ~~^~~
ZeroDivisionError: division by zero



In [49]:
# NBVAL_CHECK_OUTPUT

try:
    x.get()
except Exception as e:
    print("Error")

Error


In [50]:
# x = client.rpc_sync("div", [1, 0])

In [51]:
# NBVAL_CHECK_OUTPUT


client.rpc_sync("add", [1, 1])

2

In [52]:
def f(x, y):
    return x + y


y = client.rpc_async_fn(f, [1, 2.0])

In [53]:
y.get()

3.0

In [54]:
# NBVAL_CHECK_OUTPUT

client.rpc_sync_fn(f, [3.0, 2.0])

5.0

In [55]:
tp = []
N = 30
for i in range(N):
    fn = random.choice(("add", "mul", "div", "lista", "tupla", "dic"))
    t = (fn, [random.random(), random.random()], {})
    tp.append(t)

t = ("sleep", [10], {})
tp.append(t)
tp


[('div', [0.8168862352949218, 0.7333809295003134], {}),
 ('dic', [0.6973236974343471, 0.9989300527117113], {}),
 ('dic', [0.41264142897586653, 0.2655888940555273], {}),
 ('add', [0.02610527599495216, 0.13990399889640115], {}),
 ('tupla', [0.37080212620240205, 0.23445746884191665], {}),
 ('dic', [0.021652189882378825, 0.8621451097160423], {}),
 ('div', [0.6254734335968059, 0.44000910705065976], {}),
 ('tupla', [0.07725822525921955, 0.16573192978163498], {}),
 ('lista', [0.169569353285985, 0.6970603054287784], {}),
 ('mul', [0.3901198883352862, 0.007078708968836667], {}),
 ('dic', [0.40814810684713454, 0.5571275168055901], {}),
 ('mul', [0.570490782004445, 0.4737657004923449], {}),
 ('lista', [0.09046801215375266, 0.6170978186019582], {}),
 ('div', [0.9194046126627221, 0.4598119245016956], {}),
 ('add', [0.5653073897833613, 0.3961763299734782], {}),
 ('div', [0.008946513383881416, 0.6583253885671908], {}),
 ('mul', [0.1359125368228863, 0.9209506065781602], {}),
 ('tupla', [0.230330895341

In [56]:
tp = [
    ("div", [0.5273328219558507, 0.13835718442890943], {}, "cola_1"),
    ("div", [0.7224042937278776, 0.6744424742714074], {}, "cola_1"),
    ("mul", [0.16464192867054384, 0.2961055537758295], {}),
    ("lista", [0.24324779336838243, 0.4486736957376778], {}),
    ("dic", [0.222443603731179, 0.049201002693669005], {}),
    ("dic", [0.5892562777042863, 0.8190093828367946], {}),
    ("div", [0.9698052964451762, 0.2495544466990297], {}),
    ("add", [0.18281717945238662, 0.28456253134154685], {}),
    ("div", [0.8231058337900704, 0.4550312056693214], {}),
    ("mul", [0.6955981505190975, 0.2960636895338262], {}),
    ("add", [0.5793774647414703, 0.9353212122782703], {}),
    ("lista", [0.3893530442489298, 0.74231052966268], {}),
    ("div", [0.6419882052325951, 0.7661892480993476], {}),
    ("div", [0.049994104986677, 0.4562378471461709], {}),
    ("dic", [0.4734342157728231, 0.053714834674179035], {}),
    ("mul", [0.8092977961625194, 0.9195146847049076], {}),
    ("mul", [0.913778636227633, 0.6145608354175943], {}),
    ("lista", [0.7499808955353686, 0.5337360859450593], {}),
    ("dic", [0.6756209555432302, 0.9505296005797351], {}),
    ("dic", [0.5209295316393681, 0.9420323858687962], {}),
    ("div", [0.15809810362813237, 0.62619392590883], {}),
    ("dic", [0.29474022335742067, 0.893515494048087], {}),
    ("mul", [0.5349022233942071, 0.05757455715844495], {}),
    ("lista", [0.042102069906052586, 0.3469740971990074], {}),
    ("mul", [0.871021663889528, 0.007201521377221853], {}),
    ("lista", [0.6049724123450363, 0.26801461728942155], {}),
    ("dic", [0.8898534023208522, 0.5019296231298458], {}),
    ("lista", [0.9266421130454301, 0.9972178178045238], {}),
    ("mul", [0.48602513421859184, 0.199817263488683], {}),
    ("dic", [0.7163791721275343, 0.6950721561324937], {}),
    ("sleep", [10], {}),
]

In [57]:
fs = []
for t in tp:
    fs.append(client.rpc_async(*t))

In [58]:
# NBVAL_CHECK_OUTPUT

resultados = [f.get() for f in fs]
resultados

[3.8113873459661534,
 1.0711132843587332,
 0.048751389463712,
 [0.24324779336838243, 0.4486736957376778],
 {'a': 0.222443603731179, 'b': [0.222443603731179, 0.049201002693669005]},
 {'a': 0.5892562777042863, 'b': [0.5892562777042863, 0.8190093828367946]},
 3.886147128505352,
 0.4673797107939335,
 1.8088997491487095,
 0.2059413548755898,
 1.5146986770197406,
 [0.3893530442489298, 0.74231052966268],
 0.8378976954129118,
 0.10957903930898512,
 {'a': 0.4734342157728231, 'b': [0.4734342157728231, 0.053714834674179035]},
 0.7441612078707556,
 0.5615725620668042,
 [0.7499808955353686, 0.5337360859450593],
 {'a': 0.6756209555432302, 'b': [0.6756209555432302, 0.9505296005797351]},
 {'a': 0.5209295316393681, 'b': [0.5209295316393681, 0.9420323858687962]},
 0.2524746681288481,
 {'a': 0.29474022335742067, 'b': [0.29474022335742067, 0.893515494048087]},
 0.03079675863498907,
 [0.042102069906052586, 0.3469740971990074],
 0.006272681132523783,
 [0.6049724123450363, 0.26801461728942155],
 {'a': 0.8898

In [59]:
# NBVAL_CHECK_OUTPUT

fs = client.rpc_multi_async(tp)

In [60]:
time.sleep(3)

In [61]:
# NBVAL_CHECK_OUTPUT

# AsynResult.status updates the information every time it is called.
# 'PENDING' state should be assumed as transitory.
# If there are responses available in the queue or in the cache
# status or pending() updates the AsyncResult object.

[f.status for f in fs]

['OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'PENDING']

In [62]:
r = client.wait_responses(timeout=2)
r

['fs_client_1:93']

In [63]:
# NBVAL_CHECK_OUTPUT

len(r)

1

In [ ]:
time.sleep(7)

In [65]:
# NBVAL_CHECK_OUTPUT
# AsynResult.status updates information

[f.status for f in fs]

['OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK']

In [66]:
# NBVAL_CHECK_OUTPUT

client.wait_responses()

[]

In [67]:
client.pending

{}

In [68]:
# NBVAL_CHECK_OUTPUT

try:
    client.wait_responses(["kk", "qq"])
except ValueError as e:
    print(e)

wait_responses: ['kk', 'qq'] neither in responses nor in pending.


In [69]:
client._update_cache_with_all_available_responses()

In [70]:
# NBVAL_CHECK_OUTPUT

client.pending

{}

In [71]:
# NBVAL_CHECK_OUTPUT

[f.get() for f in fs]

[3.8113873459661534,
 1.0711132843587332,
 0.048751389463712,
 [0.24324779336838243, 0.4486736957376778],
 {'a': 0.222443603731179, 'b': [0.222443603731179, 0.049201002693669005]},
 {'a': 0.5892562777042863, 'b': [0.5892562777042863, 0.8190093828367946]},
 3.886147128505352,
 0.4673797107939335,
 1.8088997491487095,
 0.2059413548755898,
 1.5146986770197406,
 [0.3893530442489298, 0.74231052966268],
 0.8378976954129118,
 0.10957903930898512,
 {'a': 0.4734342157728231, 'b': [0.4734342157728231, 0.053714834674179035]},
 0.7441612078707556,
 0.5615725620668042,
 [0.7499808955353686, 0.5337360859450593],
 {'a': 0.6756209555432302, 'b': [0.6756209555432302, 0.9505296005797351]},
 {'a': 0.5209295316393681, 'b': [0.5209295316393681, 0.9420323858687962]},
 0.2524746681288481,
 {'a': 0.29474022335742067, 'b': [0.29474022335742067, 0.893515494048087]},
 0.03079675863498907,
 [0.042102069906052586, 0.3469740971990074],
 0.006272681132523783,
 [0.6049724123450363, 0.26801461728942155],
 {'a': 0.8898

In [72]:
fs = client.rpc_batch_async(tp)

In [73]:
# NBVAL_CHECK_OUTPUT

[f.get() for f in fs]

[3.8113873459661534,
 1.0711132843587332,
 0.048751389463712,
 [0.24324779336838243, 0.4486736957376778],
 {'a': 0.222443603731179, 'b': [0.222443603731179, 0.049201002693669005]},
 {'a': 0.5892562777042863, 'b': [0.5892562777042863, 0.8190093828367946]},
 3.886147128505352,
 0.4673797107939335,
 1.8088997491487095,
 0.2059413548755898,
 1.5146986770197406,
 [0.3893530442489298, 0.74231052966268],
 0.8378976954129118,
 0.10957903930898512,
 {'a': 0.4734342157728231, 'b': [0.4734342157728231, 0.053714834674179035]},
 0.7441612078707556,
 0.5615725620668042,
 [0.7499808955353686, 0.5337360859450593],
 {'a': 0.6756209555432302, 'b': [0.6756209555432302, 0.9505296005797351]},
 {'a': 0.5209295316393681, 'b': [0.5209295316393681, 0.9420323858687962]},
 0.2524746681288481,
 {'a': 0.29474022335742067, 'b': [0.29474022335742067, 0.893515494048087]},
 0.03079675863498907,
 [0.042102069906052586, 0.3469740971990074],
 0.006272681132523783,
 [0.6049724123450363, 0.26801461728942155],
 {'a': 0.8898

In [74]:
# NBVAL_CHECK_OUTPUT

client.rpc_batch_sync(tp)

[3.8113873459661534,
 1.0711132843587332,
 0.048751389463712,
 [0.24324779336838243, 0.4486736957376778],
 {'a': 0.222443603731179, 'b': [0.222443603731179, 0.049201002693669005]},
 {'a': 0.5892562777042863, 'b': [0.5892562777042863, 0.8190093828367946]},
 3.886147128505352,
 0.4673797107939335,
 1.8088997491487095,
 0.2059413548755898,
 1.5146986770197406,
 [0.3893530442489298, 0.74231052966268],
 0.8378976954129118,
 0.10957903930898512,
 {'a': 0.4734342157728231, 'b': [0.4734342157728231, 0.053714834674179035]},
 0.7441612078707556,
 0.5615725620668042,
 [0.7499808955353686, 0.5337360859450593],
 {'a': 0.6756209555432302, 'b': [0.6756209555432302, 0.9505296005797351]},
 {'a': 0.5209295316393681, 'b': [0.5209295316393681, 0.9420323858687962]},
 0.2524746681288481,
 {'a': 0.29474022335742067, 'b': [0.29474022335742067, 0.893515494048087]},
 0.03079675863498907,
 [0.042102069906052586, 0.3469740971990074],
 0.006272681132523783,
 [0.6049724123450363, 0.26801461728942155],
 {'a': 0.8898

In [75]:
tp = []
N = 30
for i in range(N):
    fn = random.choice(("add", "mul", "div", "fake"))
    t = (fn, [random.random(), random.random()], {})
    tp.append(t)

tp

[('fake', [0.7286016237796799, 0.522072827421612], {}),
 ('mul', [0.291878689242711, 0.47546686165965857], {}),
 ('div', [0.933003748450297, 0.9468596850122842], {}),
 ('div', [0.4667819043093394, 0.6960488834490345], {}),
 ('mul', [0.461135589483306, 0.7070940178800568], {}),
 ('mul', [0.6304439496349454, 0.4998964921476481], {}),
 ('mul', [0.3952091558489286, 0.38375214737059826], {}),
 ('mul', [0.14914812597827165, 0.02068086624655352], {}),
 ('div', [0.34322846285124, 0.7831077222920902], {}),
 ('fake', [0.531616842251809, 0.8944137666050002], {}),
 ('div', [0.4852931414501539, 0.4675404051436204], {}),
 ('add', [0.4024418805580413, 0.26007283887321253], {}),
 ('fake', [0.0012277112448202399, 0.8131144765464207], {}),
 ('add', [0.2010995589879272, 0.08770498283469164], {}),
 ('mul', [0.7852813424366869, 0.04313292420787873], {}),
 ('mul', [0.9423925026702624, 0.4430077044523548], {}),
 ('mul', [0.8946414637312871, 0.14525038989023686], {}),
 ('mul', [0.015482448030808338, 0.2689146

In [76]:
tp = [
    ("mul", [0.7355407026565478, 0.023519777553893007], {}),
    ("fake", [0.2520522260048764, 0.28054227055242864], {}),
    ("mul", [0.8838106264839363, 0.19639888038506714], {}),
    ("mul", [0.8857412900943293, 0.1253016179972587], {}),
    ("add", [0.0005226378683395039, 0.6568308199617323], {}),
    ("add", [0.15476536347494607, 0.4492869825171584], {}),
    ("fake", [0.7631067253940333, 0.019049359538004906], {}),
    ("fake", [0.4310102131915736, 0.675491507770126], {}),
    ("fake", [0.7140511506543913, 0.7837833237953286], {}),
    ("add", [0.0909525538014786, 0.44959184616881276], {}),
    ("add", [0.6627327150574825, 0.7401973301261011], {}),
    ("div", [0.21232540669537237, 0.7667374101737603], {}),
    ("add", [0.887254961441083, 0.21290364755712576], {}),
    ("fake", [0.7372649371990656, 0.3796617846297834], {}),
    ("add", [0.30864649241428155, 0.8777033159855755], {}),
    ("div", [0.4997997676145608, 0.45884184399530026], {}),
    ("fake", [0.0011733893324340494, 0.6016126834507816], {}),
    ("mul", [0.9307150630961242, 0.2943025202085412], {}),
    ("div", [0.6545197868355805, 0.11276464241684814], {}),
    ("mul", [0.6573680105483782, 0.13653818666573825], {}),
    ("add", [0.7959397704608381, 0.41576468218269147], {}),
    ("mul", [0.16347738503061415, 0.04898545483226813], {}),
    ("fake", [0.7815677830141265, 0.013945854620984632], {}),
    ("fake", [0.8020187012792446, 0.25875661742111045], {}),
    ("fake", [0.9043915109893543, 0.6876522434184933], {}),
    ("add", [0.929922781635924, 0.9540258004221696], {}),
    ("fake", [0.961238888823737, 0.4030318469598062], {}),
    ("div", [0.7466755564012741, 0.799944178434153], {}),
    ("mul", [0.15308290555092918, 0.45297088397324015], {}),
    ("mul", [0.3505532310800745, 0.5551384076337974], {}),
]

In [77]:
client.check_registry = "never"
client.set_default_queue("cola_1")

fs = client.rpc_batch_async(tp)

In [78]:
# NBVAL_CHECK_OUTPUT

[f.safe_get() for f in fs]

[0.017299753708316164,
 None,
 0.17357941751386985,
 0.11098481677579876,
 0.6573534578300718,
 0.6040523459921044,
 None,
 None,
 None,
 0.5405443999702914,
 1.4029300451835836,
 0.27692063003323986,
 1.1001586089982087,
 None,
 1.1863498083998572,
 1.0892637063407844,
 None,
 0.2739117886652408,
 5.804299759281539,
 0.08975583613233945,
 1.2117044526435294,
 0.008008014060514455,
 None,
 None,
 None,
 1.8839485820580935,
 None,
 0.9334095759817276,
 0.06934209904859642,
 0.1946055624926752]

In [79]:
# NBVAL_CHECK_OUTPUT

client.responses

{'fs_client_1:24': {'result': 21,
  'id': 'fs_client_1:24',
  'metadata': {'queue': 'requests_fs_server_7',
   'worker': 'fs_server_7',
   'method': 'add',
   'timing': {'request_sent': 1774951121.444273,
    'execution_start': 1774951121.5264406,
    'execution_finish': 1774951121.5264533}},
  'finished_time': 1774951121.6093092}}

In [80]:
client.check_registry = "cache"

In [81]:
# NBVAL_CHECK_OUTPUT

try:
    x = client.rpc_batch_sync(tp, timeout=5)
except Exception as e:
    print(e)

In [82]:
client.check_registry = "never"
client.set_default_queue("cola_1")

x = client.rpc_async("kk", [1, 0])

In [83]:
# NBVAL_CHECK_OUTPUT

try:
    x.get()
except Exception as e:
    print(e)

Error -32601 : The method does not exist/is not available.

 NoneType: None



In [84]:
y = client.rpc_async("add", [1, 0])

In [85]:
# NBVAL_CHECK_OUTPUT

y.get(5)

1

In [86]:
# NBVAL_CHECK_OUTPUT


def f(x, y):
    return x + y


# "never" use queue "default" don't work for rpc_async_fn or rpc_sync_fn
client.check_registry = "never"
y = client.rpc_async_fn(f, [1, 2.0])
try:
    print(y.get())
except Exception as e:
    print(e)

Error -32601 : The method does not exist/is not available.

 NoneType: None



In [87]:
client.check_registry = "cache"
y = client.rpc_async_fn(f, [1, 2.0])

In [88]:
# NBVAL_CHECK_OUTPUT

y.get()

3.0

In [89]:
# NBVAL_CHECK_OUTPUT

[k for k in client.responses.keys()]

['fs_client_1:24']

In [90]:
client.clean_used()

In [91]:
# NBVAL_CHECK_OUTPUT

[k for k in client.responses.keys()]

['fs_client_1:24']

In [92]:
# client.rpc_sync_fn(print, ["hola"])

In [93]:
# NBVAL_CHECK_OUTPUT

tp = [
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
    ("sleep", [15], {}),
]
tp

[('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {}),
 ('sleep', [15], {})]

In [94]:
# NBVAL_CHECK_OUTPUT

client.default_queue_ref

'requests_cola_1'

In [95]:
client.check_registry = "never"
fs = client.rpc_multi_async(tp, retry=True)

In [96]:
# NBVAL_CHECK_OUTPUT

gather(fs, 30, 5)

In [97]:
s = [f.status for f in fs]

In [98]:
# NBVAL_CHECK_OUTPUT

len(s), "OK" in s

(10, True)

In [99]:
[
    (time.time() if f.finished_time is None else f.finished_time) - f.creation_time
    for f in fs
]

[15.118884086608887,
 15.661766767501831,
 30.166456937789917,
 30.75185489654541,
 31.18385076522827,
 31.165476322174072,
 31.148025274276733,
 31.131069660186768,
 31.112555265426636,
 31.09458303451538]

In [100]:
[f.finished_time for f in fs]

[1774951186.1751962,
 1774951186.7406888,
 1774951201.265861,
 1774951201.8755276,
 None,
 None,
 None,
 None,
 None,
 None]

In [101]:
client.pending

{'fs_client_1:224': 1774951171.1456535,
 'fs_client_1:225': 1774951171.1640294,
 'fs_client_1:226': 1774951171.1814802,
 'fs_client_1:227': 1774951171.1984372,
 'fs_client_1:228': 1774951171.21695,
 'fs_client_1:229': 1774951171.234921}

In [102]:
fs[3].status

'OK'

In [103]:
fs[4].retry()

True

In [104]:
# [f.retry() for f in fs if not f.done()]
# no hace falta chequear si está pendiente.
[f.retry() for f in fs]

[False, False, False, False, True, True, True, True, True, True]

In [105]:
client.check_registry = "always"

In [106]:
# NBVAL_CHECK_OUTPUT

sorted(client.connector.all_queues_for_method("info"))

['requests_cola_0']

In [107]:
client.update_registry_cache()

In [108]:
# NBVAL_CHECK_OUTPUT

client.check_registry = "Never"
client._all_queue_refs_for_method("hola")

['requests_cola_1']

In [109]:
# NBVAL_CHECK_OUTPUT

client.check_registry = "always"
a = client.rpc_async("info")

In [110]:
# client.rpc_sync("div", [2, 1], timeout=10)

In [111]:
x = a.get()
x

('fs_server_7',
 {'requests_cola_0': {'info'},
  'requests_cola_1': {'add', 'dic', 'div', 'lista', 'mul', 'sleep', 'tupla'},
  'requests_cola_2': {'hola'},
  'requests_fs_server_7': {'add',
   'dic',
   'div',
   'eval_py_function',
   'hola',
   'info',
   'lista',
   'mul',
   'sleep',
   'tupla'},
  'requests_py_eval': {'eval_py_function'}})

In [112]:
# NBVAL_CHECK_OUTPUT

len(x), len(x[1])

(2, 5)

In [113]:
# NBVAL_CHECK_OUTPUT

x[1]["requests_cola_1"]

{'add', 'dic', 'div', 'lista', 'mul', 'sleep', 'tupla'}

In [114]:
# NBVAL_CHECK_OUTPUT

x[1]["requests_cola_2"]

{'hola'}

In [115]:
# NBVAL_CHECK_OUTPUT

x[1]["requests_py_eval"]

{'eval_py_function'}

In [116]:
# NBVAL_CHECK_OUTPUT

client.default_queue_ref

'requests_cola_1'

In [117]:
client.set_default_queue("cola_1")

In [118]:
# NBVAL_CHECK_OUTPUT

client.default_queue_ref

'requests_cola_1'

In [119]:
if LAUNCH_SERVER:
    client.rpc_sync("kill_all_processes", [], queue=MASTER_QUEUE, timeout=120)
    if platform.system() == "Windows":
        subprocess.run(
            ["taskkill", "/F", "/T", "/PID", str(server_process.pid)],
            capture_output=True,
        )

    else:
        server_process.terminate()
        server_process.wait()

In [120]:
# NBVAL_CHECK_OUTPUT

ps = client.rpc_async("list_processes", [], queue=MASTER_QUEUE)
ps.safe_get(3)

[(37596, 'worker1', 'fs_server_7'), (23956, 'worker1', 'fs_server_8')]

In [121]:
# NBVAL_CHECK_OUTPUT
try:
    y = client.rpc_async("add", [1, 0])
    y.safe_get(3)
except KeyError:
    print("Error")